# KrishiMitra soybean vision model (Colab / Kaggle)
Fine-tunes MobileNetV3-Large (or EfficientNet-B0) on ASDID + Mignoni, evaluates per class, exports ONNX.
**Runtime -> Change runtime type -> GPU (T4)** before running.

The Indian/Maharashtra soybean dataset is held-out evaluation only: never add it to training.

## 1. Get the `ml/` folder
Run `Compress-Archive ml ml.zip -Exclude .venv,data,artifacts` locally (or zip the folder without `.venv`), then upload `ml.zip` when prompted.

In [ ]:
import os, zipfile
os.chdir('/content')  # safe even if /content/ml was deleted
if not os.path.exists('/content/ml'):
    from google.colab import files
    up = files.upload()  # choose ml.zip
    zipfile.ZipFile(next(iter(up))).extractall('/content')
%cd /content/ml
!ls

## 2. Install extras (torch/torchvision already on Colab)

In [ ]:
!pip install -q onnx onnxruntime scikit-learn tqdm requests

## 3. Dryad credentials (free account -> My account -> API)
Needed for ASDID only. Typed values are not saved in the notebook.

In [ ]:
import os, getpass
os.environ['DRYAD_CLIENT_ID'] = getpass.getpass('DRYAD_CLIENT_ID: ')
os.environ['DRYAD_CLIENT_SECRET'] = getpass.getpass('DRYAD_CLIENT_SECRET: ')

## 4a. Download Mignoni insect data
Mendeley blocks Colab's IP. If this cell fails with 403, use 4b.

In [ ]:
!python download.py --dest data/raw --skip-asdid --images-per-class 1500

## 4b. Only if 4a failed: upload the Mignoni zip
Download it in your own browser from https://data.mendeley.com/datasets/bycbh73438/1 (Download all, ~860 MB), then run this cell and pick the file.

In [ ]:
from google.colab import files
up = files.upload()
name = next(iter(up))
!python download.py --dest data/raw --skip-asdid --images-per-class 1500 --mignoni-zip "{name}"

## 4c. Download ASDID (capped random subset per class; zips deleted after sampling)
Zips are 3-8 GB each; this takes a while.

In [ ]:
!python download.py --dest data/raw --skip-mignoni --images-per-class 1500

## 5. Map classes, shrink images, make train/val/test splits

In [ ]:
!python prepare_data.py --images-per-class 1500

## 6. Train (best checkpoint chosen by validation macro-F1)

In [ ]:
!python train.py --arch mobilenet_v3_large --epochs 12 --workers 2

## 7. Per-class metrics on the test split

In [ ]:
!python evaluate.py --ckpt artifacts/best.pt --csv data/splits/test.csv --out artifacts
!cat artifacts/confusion_matrix_test.csv

## 8. Export to ONNX (verifies parity with PyTorch)

In [ ]:
!python export_onnx.py --ckpt artifacts/best.pt --out artifacts

## 9. Download results
Put `soybean_vision.onnx` and `soybean_vision.json` into `backend/models/`.

In [ ]:
!cd artifacts && zip -r ../krishimitra_model.zip soybean_vision.onnx soybean_vision.json metrics_test.json confusion_matrix_test.csv train_history.json
from google.colab import files
files.download('/content/ml/krishimitra_model.zip')